# Introduction to AI APIs with OpenAI

AI APIs let you integrate large language model (LLM) capabilities directly into your code — no ML training required. In this notebook, we'll cover:

- **What the OpenAI API can do**
- **How to authenticate and make your first call**
- **Chat completions** (the core feature)
- **Tweaking outputs** with parameters
- **Practical examples**: summarization, classification, code generation

In [1]:
# Install the OpenAI Python SDK
%pip install openai

     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     ----- ---------------------------------- 0.2/1.2 MB 5.0 MB/s eta 0:00:01
     -------------------- ------------------- 0.6/1.2 MB 7.6 MB/s eta 0:00:01
     ---------------------------------- ----- 1.0/1.2 MB 8.0 MB/s eta 0:00:01
     ---------------------------------------- 1.2/1.2 MB 7.4 MB/s eta 0:00:00
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
     ---------------------------------------- 0.0/78.4 kB ? eta -:--:--
     ---------------------------------------- 78.4/78.4 kB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/204.6 kB ? eta -:--:--
     ------------------------------------- 204.6/204.6 kB 13.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\conor\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
from openai import OpenAI

# Best practice: store your key in an environment variable, never hardcode it
# You can get a key at: https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = ""

client = OpenAI()  # automatically reads OPENAI_API_KEY from the environment
print("Client ready!")

Client ready!


## How the API Works

The OpenAI API is **stateless** — every call is independent. You send a list of **messages** (a conversation history), and the model replies with the next message.

Each message has two fields:
| Field | Description |
|---|---|
| `role` | Who is speaking: `system`, `user`, or `assistant` |
| `content` | The text of the message |

- **`system`** — Sets the behavior/persona of the model
- **`user`** — Your prompt or question
- **`assistant`** — The model's reply (used when building multi-turn chats)

In [3]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is an API, in one sentence?"}
    ]
)

# The reply lives here:
print(response.choices[0].message.content)

An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and interact with each other.


## Key Parameters

| Parameter | What it does | Typical range |
|---|---|---|
| `model` | Which model to use (e.g. `gpt-4o`, `gpt-4o-mini`) | — |
| `temperature` | Creativity / randomness of the output | `0.0` (focused) → `2.0` (wild) |
| `max_tokens` | Max length of the response | 1 – 4096+ |
| `top_p` | Alternative to temperature (nucleus sampling) | 0.0 – 1.0 |
| `n` | How many responses to generate | 1, 2, 3… |

In [4]:
for temp in [0.0, 0.7, 1.5]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=temp,
        max_tokens=40,
        messages=[
            {"role": "user", "content": "Describe the color blue in one sentence."}
        ]
    )
    print(f"🌡 temperature={temp}: {resp.choices[0].message.content.strip()}\n")

🌡 temperature=0.0: Blue is a calming hue that evokes feelings of tranquility and serenity, reminiscent of clear skies and deep oceans.

🌡 temperature=0.7: Blue is a soothing hue reminiscent of clear skies and tranquil oceans, evoking feelings of calmness and serenity.

🌡 temperature=1.5: Blue is a soothing and tranquil color that evokes feelings of calmness and serenity, reminiscent of both the clear sky on a sunny day and the depths of the ocean.



## What Can the OpenAI API Do?

The same `/chat/completions` endpoint powers a huge range of tasks:

| Task | Example prompt |
|---|---|
| **Summarization** | "Summarize this article in 3 bullets: ..." |
| **Classification** | "Is this review positive, negative, or neutral?" |
| **Code generation** | "Write a Python function that parses a CSV." |
| **Translation** | "Translate this to Spanish: ..." |
| **Q&A / RAG** | Provide context + ask a question about it |
| **Structured output** | "Return a JSON object with name, age, city." |
| **Conversation / chat bots** | Multi-turn message history |

In [5]:
article = """
NASA's Artemis program aims to land the first woman and first person of color
on the Moon by the late 2020s. The program uses the Space Launch System (SLS),
the most powerful rocket ever built, along with the Orion spacecraft and a
lunar Gateway station in orbit around the Moon.
"""

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise summarizer."},
        {"role": "user",   "content": f"Summarize this in one sentence:\n\n{article}"}
    ]
)
print(resp.choices[0].message.content)

NASA's Artemis program plans to land the first woman and person of color on the Moon by the late 2020s using the powerful Space Launch System rocket, the Orion spacecraft, and a lunar Gateway station.


In [6]:
import json

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},   # forces valid JSON output
    messages=[
        {"role": "system", "content": "You output only valid JSON."},
        {"role": "user",   "content": (
            "Extract the name, city, and job title from this text as JSON:\n"
            "'Hi, I'm Maria, a software engineer based in Austin, Texas.'"
        )}
    ]
)

data = json.loads(resp.choices[0].message.content)
print(data)

{'name': 'Maria', 'city': 'Austin', 'job_title': 'software engineer'}


In [7]:
# Build a simple back-and-forth chat by appending messages manually
messages = [{"role": "system", "content": "You are a friendly Python tutor."}]

turns = [
    "What is a list comprehension?",
    "Can you show me an example that doubles even numbers?"
]

for user_msg in turns:
    messages.append({"role": "user", "content": user_msg})
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    
    assistant_reply = resp.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_reply})
    
    print(f"🧑 {user_msg}")
    print(f"🤖 {assistant_reply}\n")

🧑 What is a list comprehension?
🤖 A list comprehension is a concise way to create lists in Python. It allows you to generate a new list by applying an expression to each item in an existing iterable (like a list or a range) while optionally filtering items with a condition. 

The basic syntax of a list comprehension is:

```python
[expression for item in iterable if condition]
```

- **expression**: This is the output expression, which can include the item, transformations of the item, or other expressions.
- **item**: This represents the current value from the iterable being processed.
- **iterable**: This can be any iterable object, such as a list, tuple, or range.
- **condition** (optional): This is a filter that can be applied, allowing you to include only items that satisfy a certain condition.

### Examples:

1. **Basic List Comprehension**: Create a list of squares for numbers from 0 to 9.
    ```python
    squares = [x**2 for x in range(10)]
    print(squares)
    ```
   Output

## What's Next?

Now that you've got the basics, here's where to go deeper:

- **[OpenAI Docs](https://platform.openai.com/docs)** — full API reference
- **Streaming** — get tokens as they generate with `stream=True`
- **Vision** — pass images alongside text using `gpt-4o`
- **Embeddings** — convert text to vectors for search/similarity
- **Assistants API** — stateful agents with file search & code interpreter
- **Fine-tuning** — adapt a model to your own data

The key insight: almost any language task can be framed as a prompt. Start simple, iterate on your system prompt, and adjust `temperature` to tune creativity vs. precision.